# Lab 05 — MCP Connector

## Objective

Register, discover and invoke tools through a local MCP stdio connection.

## Introduction and concept

The tool decorators publish schemas and the client calls both tools after discovering them. Invalid input returns a protocol-level tool error.

## Real-world use case

An exam workflow consumes a topic list and an exam-duration calculation from an MCP service.

## Technology stack and dependencies

Python asyncio, MCP SDK 1.x and FastMCP. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

FastMCP server → stdio transport → ClientSession → list_tools → call_tool


## Step 1 — Implement the components

The tool decorators publish schemas and the client calls both tools after discovering them. Invalid input returns a protocol-level tool error.

In [1]:
__file__ = "lab.py"

import asyncio
import os
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.server.fastmcp import FastMCP

server = FastMCP("ExamPilot Lab Tools")

@server.tool()
def list_topics(subject: str) -> str:
    """List the supported topics of a fixture subject."""
    if subject != "SQL":
        raise ValueError("Unsupported subject")
    return "keys, joins"

@server.tool()
def exam_duration(question_count: int) -> int:
    """Compute the exam duration at two minutes per question."""
    if question_count < 1:
        raise ValueError("At least one question is required")
    return 2 * question_count

async def call_tools():
    params = StdioServerParameters(
        command=sys.executable, args=[str(Path(__file__).resolve()), "--server"],
        env={"PYTHONPATH": os.environ.get("PYTHONPATH", "")},
    )
    with open(os.devnull, "w", encoding="utf8") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                names = sorted(tool.name for tool in (await session.list_tools()).tools)
                print("Discovered tools:", names)
                assert names == ["exam_duration", "list_topics"]
                for name, payload, expected in [
                    ("list_topics", {"subject": "SQL"}, "keys, joins"),
                    ("exam_duration", {"question_count": 4}, "8"),
                ]:
                    result = await session.call_tool(name, arguments=payload)
                    assert not result.isError and result.content[0].text == expected
                    print(name, "returned", result.content[0].text)
                error = await session.call_tool("exam_duration", arguments={"question_count": 0})
                assert error.isError
                print("PASS: invalid count returned an MCP tool error")


## Step 2 — Run sample inputs and inspect actual output

Discover the registered tools, list SQL topics, calculate duration for four questions, then send an invalid zero count.

In [2]:
def main():
    if "--server" in sys.argv:
        server.run(transport="stdio")
    else:
        asyncio.run(call_tools())

await call_tools()

Discovered tools: ['exam_duration', 'list_topics']
list_topics returned keys, joins
exam_duration returned 8
PASS: invalid count returned an MCP tool error


## Testing and validation

Discover the registered tools, list SQL topics, calculate duration for four questions, then send an invalid zero count. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The server exposes fixture data over a local process. Database access and external transport require additional deployment configuration.

## How this relates to ExamPilot

Implemented integration. ExamPilot registers exam tools in backend/app/mcp/server.py and invokes them from backend/app/graph/mcp_node.py. This lab uses the MCP 1.x API in its own environment.

## Conclusion

Register, discover and invoke tools through a local MCP stdio connection. Use the trace and assertions to explain which decisions the workflow makes.